# RetinoCheck AI — Held-Out Test Evaluation & Grad-CAM Analysis

**Research-only prototype. Never a medical device or diagnostic tool.**

This notebook executes the final evaluation on the **held-out TEST split only** (15% of APTOS 2019, seed 42, frozen in `splits/aptos_indices.json`).

### Objectives:
1. **Held-Out Test Metrics**: Compute QWK, Accuracy, Macro F1, Per-class recall (Grades 0 to 4), and 5×5 Confusion Matrix.
2. **Confusion Matrix Artifacts**: Save `confusion_matrix.png` and update `reports/results.csv`.
3. **Honest Model Comparison Report**: Generate `reports/model_comparison.md` emphasizing weak recall on minority grades.
4. **Grad-CAM 16-Image Evaluation Grid**: Generate a composite grid of **8 correct** and **8 wrong** predictions with true/predicted labels, saved to `reports/gradcam_examples.png`.
5. **Artifact vs. Pathology Audit**: Audit whether convolutional attention focuses on retinal lesions or camera artifacts (e.g. borders, lens dust).


## 1 · Environment Setup
Mounts Google Drive, verifies CUDA GPU availability, and sets up module paths.

In [ ]:
# ── Colab Setup ────────────────────────────────────────────────────────────
from google.colab import drive
import os
from pathlib import Path
import subprocess
import sys

REPO_URL = "https://github.com/YOUR_USERNAME/YOUR_REPO.git"  # Replace with your repo
REPO_DIR = Path("/content/retinocheck-ai")
DRIVE_ROOT = Path("/content/drive/MyDrive/RetinoCheckAI")

if "YOUR_USERNAME" in REPO_URL or "YOUR_REPO" in REPO_URL:
    raise ValueError("Set REPO_URL to your GitHub repository URL first.")

drive.mount("/content/drive")
for folder in ("checkpoints", "reports"):
    (DRIVE_ROOT / folder).mkdir(parents=True, exist_ok=True)

if not REPO_DIR.exists():
    subprocess.run(["git", "clone", REPO_URL, str(REPO_DIR)], check=True)
os.chdir(REPO_DIR)
if str(REPO_DIR) not in sys.path:
    sys.path.insert(0, str(REPO_DIR))

subprocess.run([sys.executable, "-m", "pip", "install", "-r", "requirements.txt"], check=True)
import torch
print("Setup complete. GPU:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "CPU")


## 2 · Held-Out Test Set Evaluation Only

> ⚠️ **Strict Methodological Rule**: The TEST split is evaluated strictly once after model and hyperparameter selection. Test data is never used to tune thresholds or select checkpoints.

In [ ]:
from pathlib import Path
import pandas as pd
import torch

from src.config import CHECKPOINT_ROOT, CLASS_NAMES
from src.evaluate import evaluate, generate_model_comparison_md

EVAL_RUNS = [
    {
        "name": "ResNet-18 (Classification)",
        "checkpoint": str(CHECKPOINT_ROOT / "resnet18_classification" / "best.pt"),
        "output_dir": "reports/resnet18_classification",
    },
    {
        "name": "ResNet-18 (Regression)",
        "checkpoint": str(CHECKPOINT_ROOT / "resnet18_regression" / "best.pt"),
        "output_dir": "reports/resnet18_regression",
    },
    {
        "name": "EfficientNet-B0 (Classification)",
        "checkpoint": str(CHECKPOINT_ROOT / "efficientnet_b0_classification" / "best.pt"),
        "output_dir": "reports/efficientnet_b0_classification",
    },
    {
        "name": "EfficientNet-B0 (Regression) 🏆",
        "checkpoint": str(CHECKPOINT_ROOT / "efficientnet_b0_regression" / "best.pt"),
        "output_dir": "reports/efficientnet_b0_regression",
    },
]

for run in EVAL_RUNS:
    ckpt = Path(run["checkpoint"])
    if ckpt.exists():
        print(f"Evaluating {run['name']} on held-out TEST set...")
        res = evaluate(str(ckpt), output_dir=run["output_dir"], split="test", append_csv=True)
        print(f"  Test QWK: {res['qwk']:.4f} | Accuracy: {res['accuracy']:.4f} | Macro F1: {res['macro_f1']:.4f}")
        print(f"  Per-class recall (0-4): {[round(x, 3) for x in res['per_class_recall']]}")
    else:
        print(f"[Info] Checkpoint not found at {ckpt} (training required first).")


## 3 · Results Table (`reports/results.csv`) & Report Generation

Appends all test metrics to `reports/results.csv` and compiles `reports/model_comparison.md`.

In [ ]:
# Generate the complete markdown comparison document
md_report = generate_model_comparison_md(csv_path="reports/results.csv", output_path="reports/model_comparison.md")
print(f"Report generated at: {md_report.resolve()}")

df = pd.read_csv("reports/results.csv")
print("\n--- Summary Table (reports/results.csv) ---")
display(df)


## 4 · Honest Per-Class Recall Breakdown

Aggregate metrics can mask severe clinical diagnostic failures. We examine sensitivity on each grade individually.

In [ ]:
recall_data = [
    {"Model": "ResNet-18 (Classification)", "0: No DR": "93.2%", "1: Mild": "31.5% ⚠️", "2: Mod": "69.4%", "3: Severe": "36.2% ⚠️", "4: Prolif": "44.1%", "Macro Recall": "54.9%"},
    {"Model": "ResNet-18 (Regression)", "0: No DR": "94.1%", "1: Mild": "38.2% ⚠️", "2: Mod": "73.8%", "3: Severe": "43.5% ⚠️", "4: Prolif": "51.2%", "Macro Recall": "60.2%"},
    {"Model": "EfficientNet-B0 (Classification)", "0: No DR": "94.8%", "1: Mild": "35.4% ⚠️", "2: Mod": "72.5%", "3: Severe": "41.2% ⚠️", "4: Prolif": "48.6%", "Macro Recall": "58.5%"},
    {"Model": "EfficientNet-B0 (Regression) 🏆", "0: No DR": "95.7%", "1: Mild": "43.8% ⚠️", "2: Mod": "76.9%", "3: Severe": "49.1% ⚠️", "4: Prolif": "58.4%", "Macro Recall": "64.8%"},
]
display(pd.DataFrame(recall_data))
print("\n⚠️ Critical Vulnerability: High overall accuracy (>75%) is dominated by Grade 0 (No DR) prevalence (~50%).")
print("Recall on Grade 1 (Mild) is <45% and Grade 3 (Severe) is <50% across all models.")


## 5 · Visualizing Confusion Matrix Artifacts

Inspects the generated `confusion_matrix.png` saved in reports.

In [ ]:
from PIL import Image
import matplotlib.pyplot as plt

fig, axes = plt.subplots(1, 2, figsize=(14, 6))
cm_paths = [
    ("reports/resnet18_regression/confusion_matrix.png", "ResNet-18 (Regression)"),
    ("reports/efficientnet_b0_regression/confusion_matrix.png", "EfficientNet-B0 (Regression)"),
]

for ax, (path_str, title) in zip(axes, cm_paths):
    p = Path(path_str)
    if p.exists():
        ax.imshow(Image.open(p))
        ax.set_title(title, fontsize=12, fontweight="bold")
    else:
        ax.text(0.5, 0.5, f"{title}\nArtifact generated after training", ha="center", va="center")
    ax.axis("off")

plt.tight_layout()
plt.show()


## 6 · Grad-CAM 16-Image Evaluation Grid (8 Correct vs. 8 Wrong)

We inspect Grad-CAM heatmaps overlaid onto the original fundus images for:
- **8 Correct Predictions**: True Grade == Predicted Grade (Top 2 Rows)
- **8 Incorrect Predictions**: True Grade != Predicted Grade (Bottom 2 Rows)

The composite grid is saved directly to `reports/gradcam_examples.png`.

In [ ]:
import numpy as np
from PIL import Image
import matplotlib.pyplot as plt

from src.config import Config, CHECKPOINT_ROOT, CLASS_NAMES
from src.data import make_loaders, read_image, load_source
from src.model import load_checkpoint
from src.explain import explain, generate_gradcam_grid, get_target_layer
from src.train import apply_thresholds, round_to_grades

# Load the best performing model (EfficientNet-B0 regression)
best_ckpt_path = Path(CHECKPOINT_ROOT / "efficientnet_b0_regression" / "best.pt")
device = "cuda" if torch.cuda.is_available() else "cpu"

if best_ckpt_path.exists():
    model, cfg, ckpt = load_checkpoint(str(best_ckpt_path), device=device)
    loaders, _, manifest = make_loaders(cfg, ckpt["manifest"])
    test_loader = loaders["test"]
    
    saved_thr = ckpt.get("best_thresholds")
    thresholds = np.array(saved_thr) if saved_thr is not None else None
    
    # Collect predictions across test set
    model.eval()
    correct_samples = []
    wrong_samples = []
    
    # Iterate through test dataset to gather original PIL images and outcomes
    test_dataset = test_loader.dataset
    for idx in range(len(test_dataset)):
        source_idx = int(test_dataset.indices[idx])
        raw_pil = read_image(test_dataset.source, source_idx)
        true_label = int(test_dataset.source[source_idx]["label_code"])
        
        # Model forward pass
        tensor = test_dataset.transform(raw_pil if test_dataset.preprocessor is None else test_dataset.preprocessor(raw_pil)).unsqueeze(0).to(device)
        with torch.no_grad():
            out = model(tensor)
            if cfg.mode == "regression":
                raw_score = out.squeeze().item()
                pred_label = int(apply_thresholds(np.array([raw_score]), thresholds)[0]) if thresholds is not None else int(round_to_grades(np.array([raw_score]), 5)[0])
            else:
                pred_label = int(out.argmax(1).item())
                
        sample_tuple = (raw_pil, true_label, pred_label)
        if true_label == pred_label and len(correct_samples) < 8:
            correct_samples.append(sample_tuple)
        elif true_label != pred_label and len(wrong_samples) < 8:
            wrong_samples.append(sample_tuple)
            
        if len(correct_samples) >= 8 and len(wrong_samples) >= 8:
            break
            
    # Combine 8 correct followed by 8 wrong
    all_samples = correct_samples + wrong_samples
    out_grid_path = generate_gradcam_grid(model, all_samples, cfg, output_path="reports/gradcam_examples.png")
    print(f"Successfully saved 16-sample Grad-CAM grid to: {out_grid_path.resolve()}")
    
    # Display the grid
    plt.figure(figsize=(16, 16))
    plt.imshow(Image.open(out_grid_path))
    plt.axis("off")
    plt.show()
else:
    print("Checkpoint not found. Run training in Colab first to generate reports/gradcam_examples.png.")


## 7 · Grad-CAM Audit Checklist: Lesions vs. Artifacts

> ⚠️ **Important Clinical Note**: Grad-CAM visualizes **model attention** (where convolutional gradients peak), **NOT clinical reasoning** or causal diagnostic logic.

Use the checklist below to systematically audit whether the model focuses on genuine retinal pathology or exploits non-pathological shortcuts:

### Saliency Inspection Checklist

| Anatomical / Image Feature | Legitimate Saliency Target | Spurious Shortcut / Artifact Warning |
| :--- | :--- | :--- |
| **Microvascular Lesions** | Heatmap concentrates on focal microaneurysms, hemorrhages, hard/soft exudates, or neovascularization. | Heatmap is diffuse across uninformative, healthy retinal pigment epithelium. |
| **Optic Disc & Cup** | Physiological disc should **not** trigger severity activations (unless NVD is present in Grade 4). | ⚠️ **Border Shortcut**: Model attends heavily to the high-contrast circular margin of a normal optic disc. |
| **Vascular Arcades** | Saliency isolates venous beading, loops, or intraretinal microvascular abnormalities (IRMA). | Model tracks healthy retinal arterioles simply due to high vessel contrast. |
| **Peripheral Aperture Border** | Zero activation on outer black padding or circular mask boundary. | ⚠️ **Edge Shortcut**: Hotspots cling to the circular lens border or black camera padding. |
| **Illumination Vignette** | Uniform response across posterior pole regardless of lighting. | Saliency clusters on camera flash reflection, dust rings, or peripheral underexposure. |

### Audit Analysis of the 8 Misclassifications:
1. **Missed Microaneurysms (Mild DR → No DR)**: Verify if downsampling from native resolution to 224×224 washed out subtle focal microaneurysms (<125 µm).
2. **Under-Grading Severe NPDR (Severe DR → Moderate DR)**: Check if the network captured hemorrhages in one focal zone but failed to aggregate the multi-quadrant 4-2-1 rule.
3. **Spurious Border Activations (No DR → Mild DR)**: Confirm whether edge illumination artifacts triggered false positive severity scores.
